# BirdCLEF+ 2026 — EfficientNet v1 Training

Self-contained training notebook (no external code Dataset needed).

| Setting | Value |
|---------|-------|
| Model | tf_efficientnet_b3_ns + GeM Pooling |
| Loss | BCEWithLogitsLoss |
| Audio | hop_length=512, n_mels=128 |
| Epochs | 5 (early_stopping=3) |
| Fold | 0 only |
| GPU | T4 x2, Internet ON |

In [ ]:
import os, sys, random, ast, time, logging, pickle, warnings, types
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CONFIG
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP_DATA = p
        break
assert COMP_DATA, 'Competition data not found'

cfg = types.SimpleNamespace(
    model_name='tf_efficientnet_b3_ns', pretrained=True,
    use_gem_pooling=True, gem_p=3.0,
    num_epochs=5, batch_size=32,
    learning_rate_backbone=1e-4, learning_rate_head=1e-3,
    weight_decay=1e-6, n_folds=5, seed=42, fp16=True,
    early_stopping_patience=3, grad_clip=1.0,
    sample_rate=32000, duration=5, n_mels=128, n_fft=2048,
    hop_length=512, fmin=50, fmax=16000,
    freq_mask_param=16, time_mask_param=31,
    num_freq_masks=2, num_time_masks=2,
    mixup_alpha=0.4, secondary_label_weight=0.5,
    use_focal_loss=False,
    pink_noise_prob=0.3, pink_noise_snr_db_min=5.0, pink_noise_snr_db_max=20.0,
    train_audio_dir=f'{COMP_DATA}/train_audio',
    train_metadata=f'{COMP_DATA}/train.csv',
    taxonomy=f'{COMP_DATA}/taxonomy.csv',
    output_dir='/kaggle/working',
)
os.makedirs(cfg.output_dir, exist_ok=True)

def set_seed(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Config: epochs={cfg.num_epochs}, batch={cfg.batch_size}, hop={cfg.hop_length}, mels={cfg.n_mels}')

In [ ]:
# ============================================================
# UTILS + MODEL + DATASET + TRAINING LOOP (all in one cell)
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --- Audio utils ---
def load_audio(path, sr, duration, offset=None):
    target_samples = int(sr * duration)
    if not os.path.exists(path):
        return np.zeros(target_samples, dtype=np.float32)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            waveform, _ = librosa.load(path, sr=sr, mono=True)
    except Exception:
        return np.zeros(target_samples, dtype=np.float32)
    if len(waveform) == 0:
        return np.zeros(target_samples, dtype=np.float32)
    if offset is not None:
        start = max(0, min(int(offset * sr), len(waveform) - 1))
    elif len(waveform) > target_samples:
        start = random.randint(0, len(waveform) - target_samples)
    else:
        start = 0
    clip = waveform[start:start + target_samples]
    if len(clip) < target_samples:
        clip = np.pad(clip, (0, target_samples - len(clip)))
    return clip.astype(np.float32)

def audio_to_melspec(waveform, sr, cfg):
    mel = librosa.feature.melspectrogram(y=waveform, sr=sr, n_mels=cfg.n_mels,
        n_fft=cfg.n_fft, hop_length=cfg.hop_length, fmin=cfg.fmin, fmax=cfg.fmax)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mn, mx = mel_db.min(), mel_db.max()
    if mx - mn > 1e-6:
        return ((mel_db - mn) / (mx - mn)).astype(np.float32)
    return np.zeros_like(mel_db, dtype=np.float32)

def padded_cmap(y_true, y_pred, padding_factor=5):
    n_classes = y_true.shape[1]
    aps = []
    for c in range(n_classes):
        idx = np.argsort(y_pred[:, c])[::-1]
        true_sorted = y_true[:, c][idx]
        padded = np.concatenate([np.ones(padding_factor, dtype=np.float32), true_sorted])
        tp_cum = np.cumsum(padded)
        prec = tp_cum / (np.arange(len(padded)) + 1)
        aps.append(np.sum(prec * padded) / (tp_cum[-1] + 1e-10))
    return float(np.mean(aps))

# --- Model ---
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p).flatten(1)

class BirdCLEFModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True, use_gem=True, gem_p=3.0):
        super().__init__()
        self.use_gem = use_gem
        if use_gem:
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                num_classes=0, global_pool='', in_chans=3)
            self.pool = GeMPooling(p=gem_p)
        else:
            self.backbone = timm.create_model(model_name, pretrained=pretrained,
                num_classes=0, in_chans=3)
            self.pool = None
        self.head = nn.Linear(self.backbone.num_features, num_classes)
    def forward(self, x):
        if self.use_gem:
            return self.head(self.pool(self.backbone.forward_features(x)))
        return self.head(self.backbone(x))
    @torch.no_grad()
    def predict(self, x):
        return torch.sigmoid(self.forward(x))

# --- Dataset ---
def _parse_secondary_labels(raw):
    raw = str(raw).strip()
    if not raw or raw in ('[]', 'nan', ''): return []
    try:
        return [s.strip() for s in ast.literal_eval(raw) if isinstance(s, str) and s.strip()]
    except (ValueError, SyntaxError): return []

def _apply_spec_augment(spec, cfg):
    _, n_mels, time_steps = spec.shape
    for _ in range(cfg.num_freq_masks):
        f = random.randint(0, cfg.freq_mask_param)
        f0 = random.randint(0, max(0, n_mels - f))
        spec[:, f0:f0+f, :] = 0.0
    for _ in range(cfg.num_time_masks):
        t = random.randint(0, cfg.time_mask_param)
        t0 = random.randint(0, max(0, time_steps - t))
        spec[:, :, t0:t0+t] = 0.0
    return spec

def _add_pink_noise(waveform, snr_min, snr_max):
    n = len(waveform)
    white = np.random.randn(n).astype(np.float32)
    freqs = np.fft.rfftfreq(n); freqs[0] = 1.0
    pink_filter = 1.0 / np.sqrt(freqs); pink_filter[0] = 0.0
    pink = np.fft.irfft(np.fft.rfft(white) * pink_filter, n=n).astype(np.float32)
    sig_rms = np.sqrt(np.mean(waveform**2)) + 1e-9
    noise_rms = np.sqrt(np.mean(pink**2)) + 1e-9
    scale = sig_rms / (noise_rms * (10 ** (np.random.uniform(snr_min, snr_max) / 20.0)))
    return np.clip(waveform + scale * pink, -1.0, 1.0).astype(np.float32)

class BirdCLEFDataset(Dataset):
    def __init__(self, df, cfg, le, mode='train', secondary_label_map=None):
        self.df = df.reset_index(drop=True)
        self.cfg, self.le, self.mode = cfg, le, mode
        self.num_classes = len(le.classes_)
        self.sec_map = secondary_label_map or {}
        normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        self.transforms = transforms.Compose([transforms.RandomHorizontalFlip(0.5), normalize]) if mode == 'train' else transforms.Compose([normalize])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.cfg.train_audio_dir, row['filename'])
        waveform = load_audio(path, self.cfg.sample_rate, self.cfg.duration, None if self.mode == 'train' else 0.0)
        if self.mode == 'train' and random.random() < self.cfg.pink_noise_prob:
            waveform = _add_pink_noise(waveform, self.cfg.pink_noise_snr_db_min, self.cfg.pink_noise_snr_db_max)
        mel = audio_to_melspec(waveform, self.cfg.sample_rate, self.cfg)
        spec = torch.from_numpy(mel).unsqueeze(0)
        if self.mode == 'train': spec = _apply_spec_augment(spec, self.cfg)
        spec = self.transforms(spec.repeat(3, 1, 1))
        label = torch.zeros(self.num_classes, dtype=torch.float32)
        try: label[self.le.transform([row['primary_label']])[0]] = 1.0
        except ValueError: pass
        if self.mode == 'train' and self.cfg.secondary_label_weight > 0:
            for sl in _parse_secondary_labels(row.get('secondary_labels', '[]')):
                sl_id = self.sec_map.get(sl, sl)
                try:
                    si = self.le.transform([sl_id])[0]
                    if label[si] < self.cfg.secondary_label_weight: label[si] = self.cfg.secondary_label_weight
                except ValueError: pass
        return spec, label

class MixupDataset(Dataset):
    def __init__(self, dataset, alpha=0.4):
        self.dataset, self.alpha = dataset, alpha
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        x1, y1 = self.dataset[idx]
        if self.alpha <= 0: return x1, y1
        x2, y2 = self.dataset[random.randint(0, len(self.dataset) - 1)]
        lam = max(float(np.random.beta(self.alpha, self.alpha)), 0.5)
        return lam * x1 + (1 - lam) * x2, lam * y1 + (1 - lam) * y2

# --- Training loop ---
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, cfg, epoch):
    model.train()
    total_loss, n_batches, t0 = 0, len(loader), time.time()
    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if cfg.fp16 and device.type == 'cuda':
            with autocast():
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(images), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 100 == 0 or (i + 1) == n_batches:
            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (n_batches - i - 1)
            print(f'  Epoch {epoch:02d} [{i+1}/{n_batches}] loss={loss.item():.4f} elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m', flush=True)
    return total_loss / n_batches

@torch.no_grad()
def validate(model, loader, criterion, device, cfg):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if cfg.fp16 and device.type == 'cuda':
            with autocast():
                logits = model(images); loss = criterion(logits, labels)
        else:
            logits = model(images); loss = criterion(logits, labels)
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    return total_loss / len(loader), padded_cmap(np.vstack(all_labels), np.vstack(all_preds))

print('All definitions OK')

In [ ]:
# ============================================================
# RUN TRAINING — Fold 0 only
# ============================================================
df = pd.read_csv(cfg.train_metadata).dropna(subset=['primary_label']).reset_index(drop=True)
print(f'Samples: {len(df)}')

sec_map = {}
if os.path.exists(cfg.taxonomy):
    tax = pd.read_csv(cfg.taxonomy)
    tax['primary_label'] = tax['primary_label'].astype(str)
    if 'species_code' in tax.columns:
        sec_map = dict(zip(tax['species_code'], tax['primary_label']))
        print(f'Secondary label map: {len(sec_map)} entries')

le = LabelEncoder()
le.fit(df['primary_label'].astype(str))
num_classes = len(le.classes_)
print(f'Classes: {num_classes}')
with open(f'{cfg.output_dir}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
train_idx, val_idx = list(skf.split(df, df['primary_label']))[0]
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

train_ds = MixupDataset(BirdCLEFDataset(train_df, cfg, le, 'train', sec_map), cfg.mixup_alpha)
val_ds = BirdCLEFDataset(val_df, cfg, le, 'val')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

model = BirdCLEFModel(cfg.model_name, num_classes, cfg.pretrained, cfg.use_gem_pooling, cfg.gem_p).to(device)
backbone_params = list(model.backbone.parameters()) + (list(model.pool.parameters()) if model.pool else [])
optimizer = Adam([{'params': backbone_params, 'lr': cfg.learning_rate_backbone},
                  {'params': model.head.parameters(), 'lr': cfg.learning_rate_head}], weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=1e-7)
scaler = GradScaler(enabled=(cfg.fp16 and device.type == 'cuda'))
criterion = nn.BCEWithLogitsLoss()

best_score, patience = -1.0, 0
for epoch in range(1, cfg.num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, cfg, epoch)
    val_loss, val_cmap = validate(model, val_loader, criterion, device, cfg)
    scheduler.step()
    lrs = [pg['lr'] for pg in optimizer.param_groups]
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_cmap={val_cmap:.4f} | lr=[{", ".join(f"{lr:.2e}" for lr in lrs)}]', flush=True)
    if val_cmap > best_score:
        best_score, patience = val_cmap, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_score': val_cmap, 'num_classes': num_classes}, f'{cfg.output_dir}/fold0_best.pth')
        print(f'  -> New best: {best_score:.4f}', flush=True)
    else:
        patience += 1
        if patience >= cfg.early_stopping_patience:
            print(f'  Early stopping at epoch {epoch}'); break
print(f'\nFold 0 Best cmAP: {best_score:.4f}')

In [ ]:
# Verify output
import glob
for f in sorted(glob.glob(f'{cfg.output_dir}/*')):
    print(f'  {os.path.basename(f):30s} {os.path.getsize(f)/1024:.1f} KB')